# **03 - Train Root Model (With Weighted Loss for Class Imbalance)**
Vision Transformer (ViT-B/16) untuk binary classification: Organic vs Inorganic

**Key Improvement:** Menggunakan weighted loss untuk handle class imbalance (Organic: 20%, Inorganic: 80%)

## **Import Library**

In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# Paths
BASE_DIR = "../data/processed/level_1_root"
SAVED_MODELS_DIR = "../saved_models"
RESULTS_DIR = "../results"

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\n✓ Setup selesai!")

## **Hyperparameters**

## **Hyperparameter**

### Tujuan

Cell ini digunakan untuk mendefinisikan seluruh **hyperparameter** yang akan digunakan selama proses pelatihan model. Hyperparameter merupakan parameter yang ditentukan sebelum proses training dimulai dan berpengaruh terhadap performa model.

### Proses

Beberapa hyperparameter yang ditentukan meliputi:

- **Batch Size**: jumlah data yang diproses dalam satu iterasi.
- **Learning Rate**: besar langkah pembaruan bobot model.
- **Epoch**: jumlah pengulangan proses pelatihan terhadap seluruh dataset.
- **Momentum**: mempercepat konvergensi optimizer.
- **Weight Decay**: mengurangi overfitting melalui regularisasi.

Jumlah iterasi dalam satu epoch dapat dihitung menggunakan persamaan

$$
Iteration=
\left\lceil
\frac{N}{B}
\right\rceil
$$

dengan:

- \(N\) = jumlah data training.
- \(B\) = batch size.

### Implementasi

Nilai hyperparameter yang telah ditentukan akan digunakan pada proses pelatihan, optimizer, dan fungsi loss sehingga seluruh konfigurasi model bersifat konsisten selama training.

In [ ]:
# Training Configuration
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

# Data paths
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "val")
TEST_DIR = os.path.join(BASE_DIR, "test")

print("="*60)
print("TRAINING CONFIGURATION")
print("="*60)
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Optimizer: SGD (momentum={MOMENTUM})")
print(f"Loss: CrossEntropyLoss dengan class weights")
print(f"Device: {device}")
print("="*60)

## **Data Augmentation & Loading**
## **Data Augmentation & Loading**

### Tujuan

Cell ini bertujuan untuk mempersiapkan data sebelum digunakan pada proses pelatihan model. Tahapan yang dilakukan meliputi augmentasi citra, normalisasi, serta pemuatan dataset menggunakan `ImageFolder` dan `DataLoader`.

### Proses

Data augmentasi dilakukan untuk meningkatkan variasi data training sehingga model memiliki kemampuan generalisasi yang lebih baik. Transformasi yang digunakan meliputi:

- Resize (224×224)
- Random Horizontal Flip
- Random Vertical Flip
- Color Jitter
- ToTensor
- Normalize

Normalisasi dilakukan menggunakan persamaan

$$
x'=\frac{x-\mu}{\sigma}
$$

dengan:

- \(x\) = nilai piksel asli.
- \(\mu\) = mean ImageNet.
- \(\sigma\) = standar deviasi ImageNet.

Setelah proses transformasi selesai, dataset dimuat menggunakan **ImageFolder**, kemudian dibagi menjadi beberapa batch menggunakan **DataLoader**.

Jumlah batch pada setiap epoch dihitung menggunakan

$$
Iteration=
\left\lceil
\frac{N}{B}
\right\rceil
$$

### Implementasi

Tahap ini menghasilkan tiga DataLoader, yaitu **training**, **validation**, dan **testing**, yang selanjutnya digunakan selama proses pelatihan model.

In [ ]:
# Image transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

print("\n→ Loading datasets...")

# Create datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, data_transforms['train'])
val_dataset = datasets.ImageFolder(VAL_DIR, data_transforms['val'])
test_dataset = datasets.ImageFolder(TEST_DIR, data_transforms['test'])

# Create dataloaders
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4),
    'test': DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
}

# Class information
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"\nClass Names: {class_names}")
print(f"Number of Classes: {num_classes}")
print(f"\nDataset Sizes:")
print(f"  Train: {len(train_dataset)} images")
print(f"  Val: {len(val_dataset)} images")
print(f"  Test: {len(test_dataset)} images")

## **Calculate Class Weights (untuk handle imbalance)**


### Tujuan

Cell ini digunakan untuk menghitung **class weights** sebagai solusi terhadap permasalahan **data imbalance**, yaitu kondisi ketika jumlah data pada setiap kelas tidak seimbang.

### Proses

Distribusi data setiap kelas dihitung terlebih dahulu, kemudian bobot kelas diperoleh menggunakan metode **balanced class weight** dari Scikit-learn.

Persamaan bobot kelas adalah

$$
w_i=
\frac{N}
{C\times n_i}
$$

dengan:

- \(w_i\) = bobot kelas ke-\(i\).
- \(N\) = jumlah seluruh data training.
- \(C\) = jumlah kelas.
- \(n_i\) = jumlah data pada kelas ke-\(i\).

Semakin sedikit jumlah data pada suatu kelas, maka bobot yang diberikan akan semakin besar.

### Implementasi

Bobot kelas kemudian dikonversi menjadi Tensor PyTorch dan digunakan pada fungsi **CrossEntropyLoss**, sehingga model memberikan perhatian yang lebih besar terhadap kelas minoritas selama proses pelatihan.

In [ ]:
# Extract labels dari training dataset
train_labels = [label for _, label in train_dataset.samples]
train_labels = np.array(train_labels)

# Count class distribution
class_counts = np.bincount(train_labels)
print("\n" + "="*60)
print("CLASS DISTRIBUTION (TRAINING SET)")
print("="*60)
for i, (class_name, count) in enumerate(zip(class_names, class_counts)):
    percentage = (count / len(train_labels)) * 100
    print(f"{class_name:15s}: {count:5d} images ({percentage:5.1f}%)")

# Calculate class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

print("\n" + "="*60)
print("CLASS WEIGHTS (untuk Loss Function)")
print("="*60)
for i, (class_name, weight) in enumerate(zip(class_names, class_weights)):
    print(f"{class_name:15s}: {weight:.4f}")

# Convert to tensor
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f"\nWeights Tensor: {class_weights_tensor}")

print("\n✓ Class weights calculated!")
print("  → Minority class (organic) gets higher weight")
print("  → Majority class (inorganic) gets lower weight")
print("  → Better recall untuk minority class")

## **Model Architecture (ViT-B/16)**


Berdasarkan beberapa penelitian terdahulu, Vision Transformer (ViT) terbukti memiliki kemampuan yang baik dalam tugas klasifikasi citra karena mampu menangkap hubungan global antar bagian gambar melalui mekanisme self-attention. Berbeda dengan CNN yang berfokus pada fitur lokal, ViT memproses citra menjadi beberapa patch sehingga mampu memahami representasi citra secara lebih menyeluruh. Selain itu, pendekatan hierarchical classification yang dipadukan dengan Vision Transformer juga terbukti mampu meningkatkan performa klasifikasi sampah. Oleh karena itu, pada penelitian ini ViT-B/16 digunakan sebagai model utama untuk melakukan klasifikasi awal antara sampah organik dan anorganik. 

### Tujuan

Cell ini digunakan untuk membangun model klasifikasi menggunakan arsitektur **Vision Transformer Base Patch 16 (ViT-B/16)** dengan memanfaatkan bobot hasil **pretrained ImageNet**.

### Proses

Vision Transformer membagi gambar menjadi beberapa patch berukuran **16×16 piksel**, kemudian setiap patch diubah menjadi embedding dan diproses menggunakan mekanisme **Self-Attention**.


Jumlah patch dihitung menggunakan persamaan

$$
N=
\frac{H\times W}
{P^2}
$$

dengan:

- \(H\) = tinggi gambar.
- \(W\) = lebar gambar.
- \(P\) = ukuran patch.

Sedangkan mekanisme Self-Attention dinyatakan sebagai

$$
Attention(Q,K,V)
=
Softmax
\left(
\frac{QK^T}
{\sqrt{d_k}}
\right)V
$$

dengan:

- \(Q\) = Query.
- \(K\) = Key.
- \(V\) = Value.

### Implementasi

Model pretrained dimuat, kemudian layer klasifikasi akhir diganti agar jumlah neuron output sesuai dengan jumlah kelas pada dataset penelitian. Selanjutnya model dipindahkan ke perangkat komputasi (CPU atau GPU).

In [ ]:
print("\n→ Loading pre-trained Vision Transformer (ViT-B/16)...")

# Load pre-trained model
model_root = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

# Modify classification head untuk 2 classes
model_root.heads.head = nn.Linear(model_root.heads.head.in_features, num_classes)

# Move ke device
model_root = model_root.to(device)

print("\n✓ Model loaded!")
print(f"Total parameters: {sum(p.numel() for p in model_root.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model_root.parameters() if p.requires_grad):,}")

## **Loss Function & Optimizer (WITH WEIGHTED LOSS)**

### Tujuan

Cell ini digunakan untuk menentukan fungsi loss dan optimizer yang akan digunakan selama proses pelatihan model.

### Proses

Fungsi loss yang digunakan adalah **Weighted CrossEntropyLoss**, sedangkan optimizer yang digunakan adalah **Stochastic Gradient Descent (SGD)** dengan momentum dan weight decay.

Persamaan Weighted Cross Entropy Loss adalah

$$
L=
-\sum_{i=1}^{C}
w_i
y_i
\log(\hat y_i)
$$

dengan:

- \(w_i\) = bobot kelas.
- \(y_i\) = label sebenarnya.
- \(\hat y_i\) = probabilitas prediksi.

Sedangkan pembaruan bobot menggunakan SGD mengikuti persamaan

$$
W_{t+1}
=
W_t
-
\eta
\nabla L
$$

dengan:

- \(W_t\) = bobot lama.
- \(\eta\) = learning rate.
- \(\nabla L\) = gradien loss.

### Implementasi

Class weight yang telah dihitung sebelumnya dimasukkan ke dalam fungsi CrossEntropyLoss. Selanjutnya optimizer SGD digunakan untuk memperbarui bobot model pada setiap iterasi pelatihan.

In [3]:
# 🎯 KEY: CrossEntropyLoss dengan class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer
optimizer = optim.SGD(
    model_root.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

print("\n" + "="*60)
print("LOSS FUNCTION & OPTIMIZER")
print("="*60)
print(f"Loss: CrossEntropyLoss with class_weights")
print(f"  → organic weight: {class_weights[0]:.4f}")
print(f"  → inorganic weight: {class_weights[1]:.4f}")
print(f"\nOptimizer: SGD")
print(f"  → Learning Rate: {LEARNING_RATE}")
print(f"  → Momentum: {MOMENTUM}")
print(f"  → Weight Decay: {WEIGHT_DECAY}")
print("="*60)

NameError: name 'nn' is not defined

## **Training Function**

### Tujuan

Cell ini digunakan untuk membuat fungsi pelatihan (**training function**) yang akan dijalankan pada setiap epoch. Fungsi ini bertanggung jawab melakukan proses forward propagation, menghitung loss, memperbarui bobot model, serta mengevaluasi performa model menggunakan data validasi.

### Proses

Setiap batch data diproses melalui tahapan berikut:

1. Mengambil data dari DataLoader.
2. Melakukan **forward propagation** untuk menghasilkan prediksi.
3. Menghitung nilai loss menggunakan CrossEntropyLoss.
4. Melakukan **backpropagation**.
5. Memperbarui bobot model menggunakan optimizer.
6. Menghitung nilai accuracy.
7. Melakukan validasi menggunakan data validation.

Forward propagation dinyatakan sebagai

$$
\hat y=f(x,W)
$$

dengan:

- \(x\) = data input.
- \(W\) = bobot model.
- \(\hat y\) = hasil prediksi model.

Sedangkan pembaruan bobot dilakukan menggunakan

$$
W_{t+1}=W_t-\eta\nabla L
$$

### Implementasi

Seluruh nilai **training loss**, **validation loss**, **training accuracy**, dan **validation accuracy** disimpan pada setiap epoch untuk digunakan pada proses evaluasi dan visualisasi hasil pelatihan.

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    """
    Train model dengan tracking train/val metrics
    """
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    training_history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 50)
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_corrects = 0
        train_samples = 0
        
        for inputs, labels in dataloaders['train']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            with torch.set_grad_enabled(True):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                loss.backward()
                optimizer.step()
            
            # Statistics
            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_samples += inputs.size(0)
        
        train_loss_avg = train_loss / train_samples
        train_acc = train_corrects.double() / train_samples
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        val_samples = 0
        
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            with torch.set_grad_enabled(False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data)
            val_samples += inputs.size(0)
        
        val_loss_avg = val_loss / val_samples
        val_acc = val_corrects.double() / val_samples
        
        # Tracking
        training_history['train_loss'].append(train_loss_avg)
        training_history['train_acc'].append(train_acc.item())
        training_history['val_loss'].append(val_loss_avg)
        training_history['val_acc'].append(val_acc.item())
        
        print(f'Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc:.4f}')
        print(f'Val Loss: {val_loss_avg:.4f} | Val Acc: {val_acc:.4f}')
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f'→ Best model updated! (Acc: {best_acc:.4f})')
    
    # Load best weights
    model.load_state_dict(best_model_wts)
    
    time_elapsed = time.time() - since
    print(f'\n✓ Training complete!\n  Time: {int(time_elapsed)}s')
    
    return model, training_history

print("✓ Training function ready!")

## **Train Model**

### Tujuan

Cell ini digunakan untuk menjalankan proses pelatihan model berdasarkan hyperparameter yang telah ditentukan sebelumnya.

### Proses

Model dilatih selama sejumlah epoch tertentu. Pada setiap epoch dilakukan proses training menggunakan data training, kemudian dilanjutkan dengan evaluasi menggunakan data validation.

Jumlah iterasi setiap epoch dihitung menggunakan

$$
Iteration=
\left\lceil
\frac{N}{B}
\right\rceil
$$

dengan:

- \(N\) = jumlah data training.
- \(B\) = batch size.

Semakin banyak epoch, semakin banyak kesempatan model mempelajari pola pada dataset. Namun penggunaan epoch yang terlalu besar dapat meningkatkan risiko **overfitting**.

### Implementasi

Cell ini memanggil fungsi `train_model()` yang telah dibuat sebelumnya. Selama proses pelatihan berlangsung, sistem akan menampilkan nilai loss dan accuracy pada setiap epoch sehingga perkembangan performa model dapat dipantau.

In [ ]:
print("\n" + "="*60)
print("STARTING TRAINING...")
print("="*60)

model_root, training_history = train_model(
    model_root,
    criterion,
    optimizer,
    num_epochs=EPOCHS
)

print("\n✓ Training selesai!")

## **Save Best Model**


### Tujuan

Cell ini bertujuan untuk menyimpan model dengan performa terbaik selama proses pelatihan sehingga model yang digunakan pada tahap inferensi merupakan model dengan kemampuan generalisasi terbaik.

### Proses

Pada setiap epoch dilakukan perbandingan nilai **validation accuracy**.

Jika

$$
Accuracy_{current}
>
Accuracy_{best}
$$

maka model akan disimpan sebagai **best model**.

Sebaliknya apabila nilai validation accuracy tidak meningkat, model sebelumnya tetap dipertahankan.

### Implementasi

Model terbaik disimpan menggunakan fungsi `torch.save()`. Pendekatan ini mencegah penggunaan model dari epoch terakhir yang belum tentu memiliki performa terbaik terhadap data validasi.

In [ ]:
model_save_path = os.path.join(SAVED_MODELS_DIR, 'root_model_best.pth')
torch.save(model_root.state_dict(), model_save_path)

print(f"✓ Model saved to: {model_save_path}")
print(f"  File size: {os.path.getsize(model_save_path) / 1e6:.1f} MB")

## **Visualize Training Curves**

### Tujuan

Cell ini digunakan untuk memvisualisasikan perkembangan performa model selama proses pelatihan melalui grafik **Loss** dan **Accuracy**.

### Proses

Grafik yang ditampilkan meliputi:

- Training Loss
- Validation Loss
- Training Accuracy
- Validation Accuracy

Loss yang semakin kecil menunjukkan bahwa kesalahan prediksi model semakin berkurang.

Sedangkan accuracy dihitung menggunakan

$$
Accuracy=
\frac{Jumlah\ Prediksi\ Benar}
{Jumlah\ Seluruh\ Data}
\times100\%
$$

### Implementasi

Grafik digunakan untuk mengevaluasi apakah model mengalami:

- konvergensi,
- underfitting,
- atau overfitting.

Perbandingan kurva training dan validation menjadi indikator utama kualitas proses pelatihan.

In [ ]:
# Prepare data
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(epochs_range, training_history['train_loss'], 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, training_history['val_loss'], 'r-', label='Val Loss', linewidth=2)
axes[0].set_title('Loss per Epoch', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(epochs_range, training_history['train_acc'], 'b-', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, training_history['val_acc'], 'r-', label='Val Acc', linewidth=2)
axes[1].set_title('Accuracy per Epoch', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves_root_model.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training curves saved!")

## **Evaluate on Test Set**


### Tujuan

Cell ini digunakan untuk mengevaluasi performa model menggunakan dataset testing yang belum pernah digunakan selama proses pelatihan maupun validasi.

### Proses

Model menghasilkan prediksi terhadap seluruh data testing, kemudian dilakukan perhitungan beberapa metrik evaluasi.

Accuracy dihitung menggunakan

$$
Accuracy=
\frac{TP+TN}
{TP+TN+FP+FN}
$$

Precision

$$
Precision=
\frac{TP}
{TP+FP}
$$

Recall

$$
Recall=
\frac{TP}
{TP+FN}
$$

F1-Score

$$
F1=
2
\cdot
\frac{Precision\times Recall}
{Precision+Recall}
$$

dengan:

- TP = True Positive.
- TN = True Negative.
- FP = False Positive.
- FN = False Negative.

### Implementasi

Seluruh metrik evaluasi dihitung menggunakan library **Scikit-learn**, kemudian ditampilkan dalam bentuk **Classification Report** untuk mengetahui performa model pada masing-masing kelas.

In [2]:
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

# Set model to eval mode
model_root.eval()

# Prediction
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        outputs = model_root(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Calculate metrics
test_accuracy = np.mean(all_preds == all_labels)
print(f"\nTest Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print(f"\nConfusion Matrix:")
print(cm)

# Classification Report
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Visualize Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix - Test Set\n(Accuracy: {test_accuracy:.4f})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix_root_model.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Evaluation complete!")


EVALUATING ON TEST SET


NameError: name 'model_root' is not defined

## **Summary**
## **Training Summary**

### Tujuan

Cell ini digunakan untuk merangkum seluruh hasil pelatihan model sehingga memudahkan proses analisis sebelum model digunakan pada tahap inferensi.

### Proses

Informasi yang dirangkum meliputi:

- Model yang digunakan.
- Jumlah epoch.
- Nilai loss terbaik.
- Accuracy terbaik.
- Accuracy pada data testing.
- Lokasi penyimpanan model.

### Implementasi

Ringkasan hasil pelatihan digunakan sebagai dokumentasi akhir proses training serta menjadi acuan dalam membandingkan performa model dengan arsitektur lain pada notebook berikutnya.

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nModel: Vision Transformer (ViT-B/16)")
print(f"Task: Binary Classification (Organic vs Inorganic)")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Optimizer: SGD (LR={LEARNING_RATE}, momentum={MOMENTUM})")
print(f"Loss: CrossEntropyLoss with class weights")
print(f"  → Organic weight: {class_weights[0]:.4f}")
print(f"  → Inorganic weight: {class_weights[1]:.4f}")

print(f"\nResults:")
print(f"  Final Train Accuracy: {training_history['train_acc'][-1]:.4f}")
print(f"  Final Val Accuracy: {training_history['val_acc'][-1]:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

print(f"\nOutput Files:")
print(f"  ✓ Model: {model_save_path}")
print(f"  ✓ Training Curves: {os.path.join(RESULTS_DIR, 'training_curves_root_model.png')}")
print(f"  ✓ Confusion Matrix: {os.path.join(RESULTS_DIR, 'confusion_matrix_root_model.png')}")
print("="*60)